In [4]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qwen-vl-utils"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'qwen-vl-utils'], returncode=0)

In [5]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

model_hf = Qwen2VLForConditionalGeneration.from_pretrained(
    "Sriram1806/NLP_Orange_Problem-How_I_Met_You",
    torch_dtype=torch.float16,
    device_map="auto",
)
processor_hf = AutoProcessor.from_pretrained("Sriram1806/NLP_Orange_Problem-How_I_Met_You")

def ask_chart(image, question):
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": (
            "You are a chart analysis assistant. "
            "When given a chart and a question, briefly identify the relevant data, "
            "then output your final answer after '<ANSWER>:'. "
            "The answer must be a number, percentage, or short phrase only."
        )}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": (
                f"Look at the chart carefully.\n"
                f"Question: {question}\n\n"
                f"Briefly identify the relevant data point, then give your answer as:\n"
                f"<ANSWER>: [value]"
            )},
        ]},
    ]
    text = processor_hf.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(msgs)
    inputs = processor_hf(text=[text], images=image_inputs, return_tensors="pt").to(model_hf.device)
    with torch.inference_mode():
        out = model_hf.generate(**inputs, max_new_tokens=96, do_sample=False,
                                temperature=None, top_p=None)
    gen = out[0][inputs["input_ids"].shape[-1]:]
    raw = processor_hf.decode(gen, skip_special_tokens=True).strip()
    return raw.split("<ANSWER>:")[-1].strip() if "<ANSWER>:" in raw else raw

from datasets import load_dataset
test_ds = load_dataset("HuggingFaceM4/ChartQA", split="test")

ex = test_ds[0]
print(f"Question : {ex['query']}")
print(f"Answer   : {ask_chart(ex['image'], ex['query'])}")

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/852 [00:00<?, ?B/s]

data/train-00000-of-00003-49492f364babfa(…):   0%|          | 0.00/219M [00:00<?, ?B/s]

data/train-00001-of-00003-7302bae5e425bb(…):   0%|          | 0.00/311M [00:00<?, ?B/s]

data/train-00002-of-00003-194c9400785577(…):   0%|          | 0.00/315M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/28299 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1920 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question : How many food item is shown in the bar graph?
Answer   : 10
